# NHL RAG Pipeline Setup

This notebook sets up a complete Retrieval-Augmented Generation (RAG) pipeline for PDF documents in the `nhl-databricks` catalog.

**Key Features:**
* **Fully Local** - No external API calls to HuggingFace Hub or other services
* **Databricks Native** - Uses Databricks AI functions and Foundation Model APIs only
* **Production Ready** - High-quality embeddings with `databricks-bge-large-en` (1024-dim)
* **Scalable** - Delta tables and Vector Search for enterprise use

## Pipeline Overview
1. **Storage Setup**: Create schema and volume for PDF storage
2. **PDF Parsing**: Extract text from PDFs using Databricks `ai_parse_document()`
3. **Text Chunking**: Split documents into manageable chunks with overlap
4. **Embedding Generation**: Generate embeddings with `databricks-bge-large-en` Foundation Model
5. **Vector Search**: Create searchable index for semantic similarity
6. **Query & Retrieval**: Find relevant context and generate responses with Databricks LLMs

## Prerequisites
* Unity Catalog enabled workspace
* Serverless compute or cluster with DBR 14.3+
* PDFs uploaded to the volume

## Embedding Model
* **databricks-bge-large-en**: 1024-dimensional embeddings, no setup required

In [0]:
%pip install langchain-text-splitters pypdf2 pdfplumber databricks-vectorsearch databricks-sdk --quiet
dbutils.library.restartPython()

## Step 1: Schema and Volume Setup

Create a dedicated schema for RAG materials and a volume for PDF storage.

In [0]:
%sql
-- Create RAG schema
CREATE SCHEMA IF NOT EXISTS `nhl-databricks`.rag
COMMENT 'Schema for RAG documents, embeddings, and vector search';

-- Create volume for PDF storage
CREATE VOLUME IF NOT EXISTS `nhl-databricks`.rag.pdf_documents
COMMENT 'Storage for PDF files used in RAG pipeline';

-- Verify creation
SHOW VOLUMES IN `nhl-databricks`.rag;

## Step 2: PDF Parsing

Two approaches for extracting text from PDFs:
1. **SQL AI Function**: Use `ai_parse_document()` for automatic extraction
2. **Python Libraries**: Use PyPDF2 or pdfplumber for custom control

In [0]:
# Extract text from PDFs using ai_parse_document() SQL function
# This approach leverages Databricks AI functions for automatic text extraction

from pyspark.sql.functions import col, explode

# List all PDFs in the volume
pdf_path = "/Volumes/nhl-databricks/rag/pdf_documents/"

# Read PDF files and parse using SQL AI function
# Note: ai_parse_document() returns a VARIANT type
df_parsed = spark.sql(f"""
  SELECT 
    _metadata.file_path as file_path,
    _metadata.file_name as file_name,
    ai_parse_document(content) as parsed_content
  FROM read_files(
    '{pdf_path}',
    format => 'binaryFile',
    pathGlobFilter => '*.pdf'
  )
""")

# Extract text from VARIANT using colon notation and concatenate elements
df_text = spark.sql(f"""
  WITH parsed_docs AS (
    SELECT 
      _metadata.file_path as file_path,
      _metadata.file_name as file_name,
      ai_parse_document(content) as parsed
    FROM read_files(
      '{pdf_path}',
      format => 'binaryFile',
      pathGlobFilter => '*.pdf'
    )
  )
  SELECT
    file_name,
    file_path,
    concat_ws('\\n\\n',
      transform(
        try_cast(parsed:document:elements AS ARRAY<VARIANT>),
        element -> try_cast(element:content AS STRING)
      )
    ) AS document_text,
    parsed:metadata AS metadata
  FROM parsed_docs
  WHERE try_cast(parsed:error_status AS STRING) IS NULL
""")

display(df_text.limit(5))

In [0]:
# Alternative approach using Python libraries for more control
import PyPDF2
import io
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Define UDF to extract text from PDF bytes
@udf(returnType=StringType())
def extract_pdf_text(pdf_bytes):
    """Extract text from PDF binary content"""
    try:
        pdf_file = io.BytesIO(pdf_bytes)
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        
        text = ""
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text += page.extract_text() + "\n\n"
        
        return text.strip()
    except Exception as e:
        return f"Error parsing PDF: {str(e)}"

# Read PDFs as binary files
df_pdfs = spark.read.format("binaryFile").load(pdf_path)

# Apply UDF to extract text
df_extracted = df_pdfs.select(
    col("path").alias("file_path"),
    col("path").substr(-50, 50).alias("file_name"),
    extract_pdf_text(col("content")).alias("document_text")
)

display(df_extracted.limit(5))

## Step 3: Text Chunking

Split documents into smaller chunks (500-1000 tokens) with overlap for better retrieval.
Overlap ensures context isn't lost at chunk boundaries.

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.functions import explode, posexplode, struct, array, lit
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, IntegerType
import hashlib

# Configure chunking parameters
CHUNK_SIZE = 800  # characters (roughly 150-200 tokens)
CHUNK_OVERLAP = 150  # overlap between chunks

# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Define UDF to chunk text
@udf(returnType=ArrayType(StringType()))
def chunk_text(text):
    """Split text into overlapping chunks"""
    if not text or len(text.strip()) == 0:
        return []
    chunks = text_splitter.split_text(text)
    return chunks

# Apply chunking (using df_text from SQL approach or df_extracted from Python approach)
# Choose one based on your parsing method
df_chunks = df_text.withColumn("chunks", chunk_text(col("document_text")))

# Explode chunks into separate rows with index
df_chunks_exploded = df_chunks.select(
    col("file_name"),
    col("file_path"),
    posexplode(col("chunks")).alias("chunk_index", "chunk_text")
).withColumn(
    "document_id",
    # Create unique document ID from file path
    lit(None).cast("string")  # Will be generated with hash
)

# Generate document IDs
from pyspark.sql.functions import md5, concat_ws, current_timestamp

df_chunks_final = df_chunks_exploded.withColumn(
    "document_id",
    md5(col("file_path"))
).withColumn(
    "created_at",
    current_timestamp()
)

print(f"Total chunks created: {df_chunks_final.count()}")
display(df_chunks_final.limit(10))

## Step 4: Embedding Generation

Generate vector embeddings for each text chunk using **Databricks Foundation Model API**.
We use `databricks-bge-large-en` which produces high-quality 1024-dimensional embeddings.

**Key Features:**
* Fully within Databricks workspace (no external API calls)
* Production-quality embeddings
* No model deployment or setup required

In [0]:
# Use Databricks Foundation Model API - databricks-bge-large-en
# This is a native Databricks embedding model (no external API calls)

from databricks.sdk import WorkspaceClient
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, FloatType
import pandas as pd
import numpy as np
import time

# Initialize Databricks client
w = WorkspaceClient()

# Configuration
MODEL_NAME = "databricks-bge-large-en"  # Databricks Foundation Model
EMBEDDING_DIMENSION = 1024  # BGE-large produces 1024-dimensional embeddings
BATCH_SIZE = 10  # Smaller batch size to avoid rate limits

print(f"Using Databricks Foundation Model: {MODEL_NAME}")
print(f"Embedding dimension: {EMBEDDING_DIMENSION}")
print("This is fully within Databricks - no external API calls\n")

# Collect texts for batch processing
texts = df_chunks_final.select("chunk_text").toPandas()["chunk_text"].tolist()
total_chunks = len(texts)

print(f"Generating embeddings for {total_chunks} chunks...")
print(f"Processing in batches of {BATCH_SIZE} (this may take a few minutes)\n")

# Generate embeddings in batches
all_embeddings = []

for i in range(0, total_chunks, BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (total_chunks + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"Processing batch {batch_num}/{total_batches} ({len(batch)} texts)...", end=" ")
    
    try:
        # Call Foundation Model API
        response = w.serving_endpoints.query(
            name=MODEL_NAME,
            input=batch
        )
        
        # Extract embeddings from response
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print("✓")
        
        # Small delay to avoid rate limits
        time.sleep(0.1)
    except Exception as e:
        print(f"Error: {e}")
        # Continue with next batch

print(f"\n✓ Generated {len(all_embeddings)} embeddings")

# Add embeddings to DataFrame
from pyspark.sql import Row
from pyspark.sql.functions import monotonically_increasing_id

# Create DataFrame with embeddings
embedding_rows = [
    Row(row_id=i, embedding=emb) 
    for i, emb in enumerate(all_embeddings)
]
embeddings_df = spark.createDataFrame(embedding_rows)

# Join embeddings back to chunks
df_embeddings = df_chunks_final.withColumn(
    "row_id",
    monotonically_increasing_id()
).join(
    embeddings_df,
    "row_id"
).drop("row_id")

print(f"✓ Added embeddings to DataFrame")
print(f"✓ Total chunks with embeddings: {df_embeddings.count()}")

# Display removed to avoid daily limit - use df_embeddings.limit(5).toPandas() if needed

## Step 5: Create Delta Table for Embeddings

Store the parsed chunks and their embeddings in a Delta table for Vector Search.

In [0]:
from pyspark.sql.functions import create_map, lit

# Prepare metadata column (can include any additional info)
df_final = df_embeddings.withColumn(
    "metadata",
    create_map(
        lit("file_path"), col("file_path"),
        lit("chunk_size"), lit(str(CHUNK_SIZE)),
        lit("model"), lit(MODEL_NAME)
    )
).select(
    "document_id",
    "file_name",
    "chunk_text",
    "chunk_index",
    "embedding",
    "metadata",
    "created_at"
)

# Write to Delta table (use backticks for catalog name with hyphen)
table_name = "`nhl-databricks`.rag.document_embeddings"

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Created and populated table: {table_name}")
print(f"✓ Total records: {df_final.count()}")

# Verify table
display(spark.table(table_name).limit(10))

## Step 6: Create Vector Search Index

Create a Databricks Vector Search index for fast similarity search.

In [0]:
from databricks.vector_search.client import VectorSearchClient

# Initialize Vector Search client
vsc = VectorSearchClient()

# Configuration
VECTOR_SEARCH_ENDPOINT = "nhl_rag_endpoint"
INDEX_NAME = "nhl-databricks.rag.document_embeddings_index"
SOURCE_TABLE = "nhl-databricks.rag.document_embeddings"
EMBEDDING_DIMENSION = 1024  # databricks-bge-large-en dimension

try:
    # Check if endpoint exists, create if not
    try:
        vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT)
        print(f"✓ Endpoint '{VECTOR_SEARCH_ENDPOINT}' exists")
    except Exception:
        print(f"Creating endpoint '{VECTOR_SEARCH_ENDPOINT}'...")
        vsc.create_endpoint(
            name=VECTOR_SEARCH_ENDPOINT,
            endpoint_type="STANDARD"
        )
        print(f"✓ Created endpoint '{VECTOR_SEARCH_ENDPOINT}'")
    
    # Create delta sync index
    print(f"Creating vector search index...")
    index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",  # or "CONTINUOUS" for auto-sync
        primary_key="document_id",
        embedding_dimension=EMBEDDING_DIMENSION,
        embedding_vector_column="embedding"
    )
    
    print(f"✓ Created index: {INDEX_NAME}")
    print("✓ Index will sync automatically from the Delta table")
    
except Exception as e:
    print(f"Note: {str(e)}")
    print("\nAlternative: Create index via UI:")
    print("1. Go to Compute > Vector Search")
    print(f"2. Create endpoint: {VECTOR_SEARCH_ENDPOINT}")
    print(f"3. Create index on table: {SOURCE_TABLE}")
    print(f"4. Set embedding column: embedding, primary key: document_id")

## Step 7: Query and Retrieval

Implement the retrieval logic to search for relevant chunks based on user queries.

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
import time
import re

# Configuration (define here to make cell self-contained)
MODEL_NAME = "databricks-bge-large-en"
VECTOR_SEARCH_ENDPOINT = "nhl_rag_endpoint"
INDEX_NAME = "nhl-databricks.rag.document_embeddings_index"

# Initialize clients
vsc = VectorSearchClient()
w = WorkspaceClient()

def retrieve_context_keyword(query: str, keyword: str, top_k: int = 5):
    """
    Retrieve chunks by directly searching the source table with keywords
    (bypasses vector search for better keyword matching)
    
    Args:
        query: User's question
        keyword: Keyword to search for (e.g., "Elo")
        top_k: Number of chunks to retrieve
    
    Returns:
        List of relevant text chunks
    """
    print(f"Searching source table for '{keyword}'...")
    
    # Query source table directly
    df = spark.table("`nhl-databricks`.rag.document_embeddings")
    
    # Filter for keyword and score by relevance
    keyword_chunks = df.filter(
        df.chunk_text.rlike(f"(?i){keyword}")
    )
    
    # Convert to pandas for scoring
    chunks_pd = keyword_chunks.select(
        "document_id", "file_name", "chunk_text", "chunk_index"
    ).toPandas()
    
    if len(chunks_pd) == 0:
        print(f"No chunks found containing '{keyword}'")
        return None
    
    # Score each chunk
    explanatory_keywords = [
        'invented', 'created', 'developed', 'designed',
        'originally', 'system', 'formula', 'calculate',
        'measure', 'rating', 'method', 'used in chess'
    ]
    
    def score_chunk(text):
        score = 0
        text_lower = text.lower()
        # Bonus for explanatory words
        for word in explanatory_keywords:
            if word in text_lower:
                score += 1
        # Prefer chunks with the exact phrase "elo rating" or "elo system"
        if 'elo rating' in text_lower or 'elo system' in text_lower:
            score += 5
        return score
    
    chunks_pd['relevance_score'] = chunks_pd['chunk_text'].apply(score_chunk)
    
    # Sort by score and take top_k
    best_chunks = chunks_pd.nlargest(top_k, 'relevance_score')
    
    print(f"✓ Found {len(chunks_pd)} chunks containing '{keyword}'")
    print(f"✓ Selected top {len(best_chunks)} most explanatory chunks")
    
    # Format as vector search result structure
    data_array = []
    for _, row in best_chunks.iterrows():
        data_array.append([
            row['document_id'],
            row['file_name'],
            row['chunk_text'],
            row['chunk_index']
        ])
    
    return {'result': {'data_array': data_array}}

def retrieve_context_vector(query: str, top_k: int = 5):
    """
    Retrieve using pure vector search
    """
    print(f"Generating embedding for query...")
    response = w.serving_endpoints.query(
        name=MODEL_NAME,
        input=[query]
    )
    query_embedding = response.data[0].embedding
    
    # Check index status
    print(f"Checking index status...")
    try:
        index = vsc.get_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=INDEX_NAME
        )
        index_status = index.describe()
        status_state = index_status.get('status', {}).get('detailed_state', 'UNKNOWN')
        
        if not status_state.startswith('ONLINE'):
            print(f"\nIndex status: {status_state}")
            return None
    except NotFound as e:
        print(f"\nIndex not found: {e}")
        return None
    
    # Search vector index
    print(f"Searching vector index...")
    results = index.similarity_search(
        query_vector=query_embedding,
        columns=["document_id", "file_name", "chunk_text", "chunk_index"],
        num_results=top_k
    )
    
    return results

def retrieve_context_hybrid(query: str, top_k: int = 5, keyword: str = None):
    """
    Hybrid search: prioritize keyword search if keyword provided
    
    Args:
        query: User's question
        top_k: Number of chunks to retrieve
        keyword: Optional keyword (if provided, uses keyword search)
    
    Returns:
        List of relevant text chunks
    """
    if keyword:
        # Use keyword search (more reliable)
        return retrieve_context_keyword(query, keyword, top_k)
    else:
        # Fall back to vector search
        return retrieve_context_vector(query, top_k)

def format_context(results):
    """
    Format retrieved chunks into a context string for LLM
    """
    if results is None:
        return ""
    
    context_parts = []
    for idx, row in enumerate(results.get('result', {}).get('data_array', []), 1):
        chunk_text = row[2]  # chunk_text is at index 2
        file_name = row[1]   # file_name is at index 1
        context_parts.append(f"[Document {idx}: {file_name}]\n{chunk_text}")
    
    return "\n\n---\n\n".join(context_parts)

# Example query with keyword-based hybrid search
test_query = "What are Elo ratings and how are they used in sports?"
print(f"Query: {test_query}\n")

# Use keyword search for better results
results = retrieve_context_hybrid(test_query, top_k=3, keyword="Elo")

if results is not None:
    context = format_context(results)
    print("\nRetrieved Context:")
    print("="*80)
    print(context)

In [0]:
# Use ai_query() SQL function to generate response with retrieved context

def generate_response(query: str, context: str):
    """
    Generate response using retrieved context and Databricks AI
    """
    # Build prompt with context
    prompt = f"""You are a helpful assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}

Answer based on the context provided. If the context doesn't contain relevant information, say so.
"""
    
    # Escape single quotes for SQL
    escaped_prompt = prompt.replace("'", "\\'")
    
    # Use SQL AI function to generate response
    response_df = spark.sql(f"""
        SELECT ai_query(
            'databricks-meta-llama-3-3-70b-instruct',
            '{escaped_prompt}'
        ) as response
    """)
    
    return response_df.collect()[0]['response']

# Complete RAG pipeline example
query = "What are Elo ratings and how are they used in sports analytics?"

print(f"Question: {query}\n")

# Step 1: Retrieve relevant context using hybrid search
# Use keyword="Elo" for keyword-based search
results = retrieve_context_hybrid(query, top_k=3, keyword="Elo")

if results is None:
    print("\nCannot proceed: Unable to retrieve context.")
else:
    context = format_context(results)
    
    print("Retrieved Documents:")
    for idx, row in enumerate(results.get('result', {}).get('data_array', []), 1):
        print(f"  {idx}. {row[1]} (chunk {row[3]})")
    
    # Step 2: Generate response
    print("\nGenerating response...\n")
    response = generate_response(query, context)
    
    print("Response:")
    print("="*80)
    print(response)

## Step 8: Complete End-to-End Pipeline

Put it all together: Process a new PDF through the entire RAG pipeline.

In [0]:
def process_pdf_to_rag(pdf_path: str, file_name: str):
    """
    Complete pipeline: PDF → Parsing → Chunking → Embedding → Storage
    
    Args:
        pdf_path: Path to the PDF file in the volume
        file_name: Name of the PDF file
    """
    from databricks.sdk import WorkspaceClient
    import time
    
    print(f"Processing: {file_name}")
    print("="*80)
    
    # Step 1: Parse PDF
    print("[1/5] Parsing PDF...")
    df_parsed = spark.sql(f"""
        WITH parsed_docs AS (
            SELECT 
                _metadata.file_path as file_path,
                _metadata.file_name as file_name,
                ai_parse_document(content) as parsed
            FROM read_files(
                '{pdf_path}',
                format => 'binaryFile'
            )
        )
        SELECT
            file_name,
            file_path,
            concat_ws('\\n\\n',
                transform(
                    try_cast(parsed:document:elements AS ARRAY<VARIANT>),
                    element -> try_cast(element:content AS STRING)
                )
            ) AS document_text
        FROM parsed_docs
        WHERE try_cast(parsed:error_status AS STRING) IS NULL
    """)
    
    # Step 2: Chunk text
    print("[2/5] Chunking text...")
    df_chunks = df_parsed.withColumn("chunks", chunk_text(col("document_text")))
    df_chunks_exploded = df_chunks.select(
        col("file_name"),
        col("file_path"),
        posexplode(col("chunks")).alias("chunk_index", "chunk_text")
    ).withColumn(
        "document_id",
        md5(concat_ws("_", col("file_path"), col("chunk_index")))
    ).withColumn(
        "created_at",
        current_timestamp()
    )
    
    num_chunks = df_chunks_exploded.count()
    print(f"   Created {num_chunks} chunks")
    
    # Step 3: Generate embeddings using Databricks Foundation Model
    print(f"[3/5] Generating embeddings using {MODEL_NAME}...")
    w = WorkspaceClient()
    
    texts = df_chunks_exploded.select("chunk_text").toPandas()["chunk_text"].tolist()
    all_embeddings = []
    
    batch_size = 10  # Smaller batch size to avoid rate limits
    for i in range(0, num_chunks, batch_size):
        batch = texts[i:i+batch_size]
        print(f"   Processing batch {i//batch_size + 1}/{(num_chunks + batch_size - 1)//batch_size}...")
        
        response = w.serving_endpoints.query(
            name=MODEL_NAME,
            input=batch
        )
        all_embeddings.extend(response.predictions)
        time.sleep(0.1)  # Small delay to avoid rate limits
    
    # Step 4: Add embeddings to DataFrame
    print("[4/5] Preparing data...")
    from pyspark.sql import Row
    
    embedding_rows = [
        Row(row_id=i, embedding=emb) 
        for i, emb in enumerate(all_embeddings)
    ]
    embeddings_df = spark.createDataFrame(embedding_rows)
    
    df_final = df_chunks_exploded.withColumn(
        "row_id",
        monotonically_increasing_id()
    ).join(
        embeddings_df,
        "row_id"
    ).drop("row_id").withColumn(
        "metadata",
        create_map(
            lit("file_path"), col("file_path"),
            lit("model"), lit(MODEL_NAME)
        )
    ).select(
        "document_id",
        "file_name",
        "chunk_text",
        "chunk_index",
        "embedding",
        "metadata",
        "created_at"
    )
    
    # Step 5: Append to Delta table
    print("[5/5] Saving to Delta table...")
    df_final.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("`nhl-databricks`.rag.document_embeddings")
    
    print(f"\n✓ Successfully processed {file_name}")
    print(f"✓ Added {num_chunks} chunks to the embeddings table")
    print(f"✓ Vector search index will sync automatically\n")
    
    return df_final

# Example usage (uncomment when you have PDFs uploaded)
# sample_pdf = "/Volumes/nhl-databricks/rag/pdf_documents/nhl_rulebook.pdf"
# result = process_pdf_to_rag(sample_pdf, "nhl_rulebook.pdf")

print("Pipeline function ready!")
print("\nTo process a PDF, run:")
print('process_pdf_to_rag("/Volumes/nhl-databricks/rag/pdf_documents/your_file.pdf", "your_file.pdf")')
print(f"\nUsing Databricks Foundation Model: {MODEL_NAME}")
print("Fully within Databricks - no external API calls.")

## Next Steps

### 1. Upload PDFs
Upload your PDF documents to the volume:
```
/Volumes/nhl-databricks/rag/pdf_documents/
```

You can upload via:
- **UI**: Catalog Explorer → nhl-databricks → rag → pdf_documents → Upload
- **Code**: `dbutils.fs.cp("local_path.pdf", "/Volumes/nhl-databricks/rag/pdf_documents/")`

### 2. Process PDFs
Run the `process_pdf_to_rag()` function for each PDF you want to add to the RAG system.

### 3. Query Your Documents
Use the `retrieve_context()` and `generate_response()` functions to ask questions about your documents.

### 4. Monitor and Maintain
- Check vector search index sync status
- Monitor embedding table growth
- Update embeddings when documents change
- Fine-tune chunk size and overlap based on your use case

### 5. Optimization Tips
- Use `CONTINUOUS` pipeline type for real-time updates
- Experiment with different embedding models (larger models = better quality, slower speed)
- Adjust `top_k` in retrieval based on your needs
- Add metadata filters for more targeted searches

---

**Your RAG pipeline is ready!** 🚀